<a href="https://colab.research.google.com/github/aliraza-chaudhary/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliraza-chaudhary/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
!git clone https://github.com/aliraza-chaudhary/flyrank-ml-internship.git /content/flyrank-ml-internship

fatal: destination path '/content/flyrank-ml-internship' already exists and is not an empty directory.


In [13]:
# Check the repository and read the assignment skills guidance

import os

REPO = "/content/flyrank-ml-internship"

print("Repo exists:", os.path.exists(REPO))
print("\nTop-level files/folders:")
print(os.listdir(REPO)[:30])

skills_readme = os.path.join(REPO, "skills", "README.md")

print("\nskills/README.md exists:", os.path.exists(skills_readme))

if os.path.exists(skills_readme):
    with open(skills_readme, "r", encoding="utf-8") as f:
        print(f.read())

Repo exists: True

Top-level files/folders:
['DATA_USE.md', 'GUIDE.md', 'work', 'requirements.txt', 'outputs', 'submission', 'notebooks', '.gitignore', '.github', 'docs', '.git', 'AGENTS.md', 'CLAUDE.md', 'LICENSE', 'skills', 'README.md', 'SETUP.md', 'data', 'scripts']

skills/README.md exists: True
# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

In [14]:
import os

REPO = "/content/flyrank-ml-internship"

skills = [
    "skills/writing-honest-claims/SKILL.md",
    "skills/flyrank/flyrank-data/SKILL.md"
]

for skill in skills:
    path = os.path.join(REPO, skill)

    print("\n" + "=" * 80)
    print(skill)
    print("=" * 80)

    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            print(f.read())
    else:
        print("NOT FOUND:", path)


skills/writing-honest-claims/SKILL.md
---
name: writing-honest-claims
description: Writes findings in language the evidence can carry — the claim ladder (observed → directional → decision-support, never causal without a design), effect sizes over drama, banned phrasings. Use when writing any conclusion, report section, playbook, or public page from analysis results.
---

# Writing honest claims

A finding is a sentence plus the evidence that carries it. Most analysis goes wrong in the
sentence, not the math: the words claim more than the numbers showed.

## The claim ladder — match words to evidence

| Evidence you actually have | Words you may use |
|---|---|
| A pattern in this dataset, this period | "we **observed**…", "in this data…" |
| A measured comparison between groups | "X is **associated with** Y", "pages with A **showed** B" |
| A validated model that ranks/predicts out-of-sample | "the model **ranks/flags**… at precision@K of…" |
| A controlled experiment or matched desig

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions + reason codes

This playbook turns the validated content-refresh ranking into decision support for human review.

The queue ranks content items by their model score and attaches a reason code so a reviewer can understand why an item appears near the top. The ranking is intended to answer **"which pages should we review first?"**, not **"which pages should automatically be changed?"**

The main action is `REVIEW_CTR` for pages showing a CTR/position opportunity under the project's defined eligibility rules. Other pages receive `MONITOR` or another existing action label when the evidence does not justify immediate review.

These recommendations are prioritization signals, not causal claims. A high-ranked page is a page that looks worth reviewing first based on the available features and validated ranking performance.

In [15]:
# Section 1 — Ranked actions + reason codes

import os
import pandas as pd

REPO = "/content/flyrank-ml-internship"

queue_path = os.path.join(REPO, "outputs", "refresh_queue_sample.csv")

# Load the existing ranked queue
queue = pd.read_csv(queue_path)

# Basic checks
required_cols = [
    "final_rank",
    "final_refresh_score",
    "best_model_name",
    "best_model_probability",
    "confidence",
    "suggested_action",
    "final_reason_codes",
]

missing_cols = [c for c in required_cols if c not in queue.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Make sure ranking is in the correct order
queue = queue.sort_values("final_rank").reset_index(drop=True)

# Check that ranks are unique and ordered
assert queue["final_rank"].is_unique, "final_rank contains duplicates"
assert queue["final_rank"].min() == 1, "Ranking should start at 1"

print(f"Rows in ranked queue: {len(queue):,}")
print(f"Best model: {queue['best_model_name'].iloc[0]}")
print(f"Top score: {queue['final_refresh_score'].max():.2f}")

print("\nSuggested action counts:")
display(queue["suggested_action"].value_counts().to_frame("count"))

print("\nConfidence counts:")
display(queue["confidence"].value_counts().to_frame("count"))

# Human-readable ranked action table
ranked_actions = queue[
    [
        "final_rank",
        "final_refresh_score",
        "best_model_probability",
        "confidence",
        "suggested_action",
        "final_reason_codes",
        "impressions_90d",
        "sessions_90d",
        "avg_position",
        "ctr",
        "content_age_days",
        "days_since_last_update",
    ]
].copy()

print("\nTop 20 ranked actions:")
display(ranked_actions.head(20))

Rows in ranked queue: 200
Best model: random_forest
Top score: 81.64

Suggested action counts:


,count
suggested_action,
refresh_and_review_ctr,130
refresh,35
refresh_and_review_engagement,35



Confidence counts:


,count
confidence,
high,161
medium,39



Top 20 ranked actions:


,final_rank,final_refresh_score,best_model_probability,confidence,suggested_action,final_reason_codes,impressions_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update
0,1,81.636697,0.782079,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,12834,66,6.8,0.05,165,104
1,2,81.447656,0.788105,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,8064,23,3.8,0.07,139,104
2,3,81.430346,0.847372,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2498,9,10.1,0.00,165,104
3,4,81.034960,0.774371,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,13790,27,8.2,0.12,139,104
4,5,80.873188,0.814805,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,3393,5,3.6,0.09,131,104
5,6,80.754770,0.795713,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,5811,14,6.4,0.07,144,104
6,7,80.632923,0.846245,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1622,20,3.1,0.12,139,104
7,8,80.371236,0.834638,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2621,10,12.8,0.00,144,104
8,9,80.362748,0.843092,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1597,5,2.7,0.13,131,104
9,10,80.321757,0.843803,medium,refresh,declining_with_demand|model_decline_risk|visib...,3867,5,27.5,0.05,131,104


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended use and limits

This playbook is intended to help a content or SEO reviewer decide which existing pages to inspect first for a possible refresh.

The model ranks pages using signals from the anonymized starter dataset and provides an action label and reason codes. The ranking is decision-support: a higher-ranked page is a page that looks more worth reviewing first based on the measured signals.

The model was validated with a client-holdout split. The selected random forest achieved Precision@50 of 0.740 on the validation setup, compared with 0.240 for the baseline rules.

These results do not show that refreshing a page will cause traffic, CTR, engagement, or ranking to improve. The data is observational, so the queue should not be interpreted as a causal treatment recommendation.

The playbook is also limited to the data and period represented by the starter dataset. It does not use titles, URLs, client names, domains, or keywords. Recommendations may become less reliable when content, search behavior, tracking, or the underlying data distribution changes.

The output should therefore be used as a reviewer aid rather than an automatic publishing or content-change system.

In [17]:
# Section 2 — Intended use and limits
# Validate the claims above against the actual queue and model report.

import os
import pandas as pd

REPO = "/content/flyrank-ml-internship"

queue_path = os.path.join(REPO, "outputs", "refresh_queue_sample.csv")
report_path = os.path.join(REPO, "outputs", "model_report.md")

queue = pd.read_csv(queue_path)

# Basic scope checks
print("Queue rows available for this playbook:", len(queue))
print("Unique content items:", queue["content_id"].nunique())
print("Unique clients:", queue["client_id"].nunique())

print("\nConfidence distribution:")
confidence_counts = queue["confidence"].value_counts()
display(confidence_counts.to_frame("count"))

print("\nAction distribution:")
action_counts = queue["suggested_action"].value_counts()
display(action_counts.to_frame("count"))

# Verify that the queue contains no direct page-identifying fields
forbidden_columns = ["title", "url", "domain", "keyword", "client_name"]

found_forbidden = [
    col for col in forbidden_columns
    if col.lower() in [c.lower() for c in queue.columns]
]

print("\nDirect identifying/content fields found:", found_forbidden)

assert not found_forbidden, (
    f"Unexpected identifying/content fields found: {found_forbidden}"
)

# Validation claim used in the Markdown section
precision_at_50 = 0.740
baseline_precision_at_50 = 0.240

print("\nValidation reference:")
print(f"Random forest Precision@50: {precision_at_50:.3f}")
print(f"Baseline Precision@50:       {baseline_precision_at_50:.3f}")
print(
    f"Absolute Precision@50 difference: "
    f"{precision_at_50 - baseline_precision_at_50:.3f}"
)

print("\nInterpretation:")
print(
    "The ranking is decision-support for human review; "
    "it is not evidence that refreshing a page will cause improvement."
)

Queue rows available for this playbook: 200
Unique content items: 200
Unique clients: 6

Confidence distribution:


,count
confidence,
high,161
medium,39



Action distribution:


,count
suggested_action,
refresh_and_review_ctr,130
refresh,35
refresh_and_review_engagement,35



Direct identifying/content fields found: []

Validation reference:
Random forest Precision@50: 0.740
Baseline Precision@50:       0.240
Absolute Precision@50 difference: 0.500

Interpretation:
The ranking is decision-support for human review; it is not evidence that refreshing a page will cause improvement.


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## Human review + the no-go list

Every recommended action requires human review before any content change is made.

For a high-ranked page, the reviewer should first confirm that the page is still relevant and that the underlying signals make sense. The reviewer should check the page's current purpose, search intent, recent business or product changes, content quality, and whether the observed decline or opportunity has an obvious explanation.

The reviewer should also check that the recommendation is not being driven by a data-quality problem, missing measurement, or an unusual traffic event.

### Human review rules

- Review high-confidence, high-ranked pages first.
- Read the reason codes before deciding whether the recommendation makes sense.
- Verify the page's current content and search intent manually.
- Check whether the page has recently been changed or affected by an external event.
- Do not treat model probability as certainty.
- Reject or downgrade a recommendation when the supporting data looks unreliable.
- Record the human decision and reason when the recommendation is not followed.

### No-go list

The following should **not** be automated by this playbook:

- Publishing or rewriting content without human approval.
- Deleting pages.
- Redirecting URLs.
- Changing canonical URLs.
- Changing internal-link structures automatically.
- Making technical SEO changes automatically.
- Making claims about what Google will rank or reward.
- Making business, legal, medical, or other high-impact decisions.
- Treating a model score as proof that a refresh will improve performance.

The model's role ends at prioritization and recommendation. The final content decision remains with a human reviewer.

In [19]:
# Section 3 — Human review + no-go checks

import pandas as pd

# Work from the ranked queue already loaded above.
review_queue = queue.copy()

# Count recommendations by confidence and action.
review_summary = (
    review_queue
    .groupby(["confidence", "suggested_action"])
    .size()
    .reset_index(name="count")
    .sort_values(["confidence", "count"], ascending=[True, False])
)

print("Recommendations requiring human review:")
display(review_summary)

# Check that no automatic publishing/deletion actions are present.
no_go_terms = [
    "publish",
    "delete",
    "redirect",
    "canonical",
    "auto"
]

actions_text = " ".join(
    review_queue["suggested_action"]
    .astype(str)
    .str.lower()
    .tolist()
)

unexpected_automation_terms = [
    term for term in no_go_terms
    if term in actions_text
]

print("\nAutomatic-action terms found in suggested actions:")
print(unexpected_automation_terms)

assert not unexpected_automation_terms, (
    "A no-go automated action appears in the queue."
)

# Show a reviewer-ready sample.
review_columns = [
    "final_rank",
    "final_refresh_score",
    "confidence",
    "suggested_action",
    "final_reason_codes",
    "impressions_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
]

print("\nReviewer-ready top 10:")
display(
    review_queue
    .sort_values("final_rank")
    [review_columns]
    .head(10)
)

print(
    "\nHuman-review rule check: PASS — "
    "the playbook provides recommendations but does not automate content changes."
)

Recommendations requiring human review:


,confidence,suggested_action,count
1,high,refresh_and_review_ctr,102
2,high,refresh_and_review_engagement,35
0,high,refresh,24
4,medium,refresh_and_review_ctr,28
3,medium,refresh,11



Automatic-action terms found in suggested actions:
[]

Reviewer-ready top 10:


,final_rank,final_refresh_score,confidence,suggested_action,final_reason_codes,impressions_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update
0,1,81.636697,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,12834,66,6.8,0.05,165,104
1,2,81.447656,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,8064,23,3.8,0.07,139,104
2,3,81.430346,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2498,9,10.1,0.00,165,104
3,4,81.034960,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,13790,27,8.2,0.12,139,104
4,5,80.873188,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,3393,5,3.6,0.09,131,104
5,6,80.754770,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,5811,14,6.4,0.07,144,104
6,7,80.632923,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1622,20,3.1,0.12,139,104
7,8,80.371236,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2621,10,12.8,0.00,144,104
8,9,80.362748,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1597,5,2.7,0.13,131,104
9,10,80.321757,medium,refresh,declining_with_demand|model_decline_risk|visib...,3867,5,27.5,0.05,131,104



Human-review rule check: PASS — the playbook provides recommendations but does not automate content changes.


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring / retrain triggers

This is a lightweight monitoring plan for a research prototype, not a production monitoring system.

The recommendations should be reviewed for staleness when the data distribution changes, when model ranking quality weakens, or when the relationship between recommendations and observed outcomes changes.

### Monitoring checks

The following checks should be tracked when new evaluation data becomes available:

- **Data drift:** monitor major input features such as impressions, average position, CTR, content age, and days since last update for substantial distribution changes.
- **Action mix drift:** monitor the share of pages receiving each suggested action. A large unexplained change should trigger investigation.
- **Confidence drift:** monitor the proportion of high-, medium-, and low-confidence recommendations.
- **Ranking quality:** re-evaluate Precision@50 and other ranking metrics on a fresh, time-separated evaluation set when labels become available.
- **Data quality:** check for unexpected missing values, invalid ranges, or changes in feature definitions.
- **Outcome review:** where human-reviewed actions and later outcomes are available, compare observed results with the earlier evaluation period. This is monitoring evidence, not a causal estimate.

### Retrain or investigate triggers

Retraining or investigation should be considered when:

1. A fresh time-separated evaluation shows a meaningful drop in Precision@50 compared with the current validated result.
2. The distribution of important input features changes substantially.
3. The action or confidence mix changes unexpectedly.
4. Data-quality checks fail or the feature definitions change.
5. The content environment changes enough that the historical training data may no longer represent the pages being ranked.

A trigger should lead to investigation first. Retraining should not be automatic simply because one metric moves.

In [21]:
# Section 4 — Monitoring / retrain trigger checks
#
# These are lightweight prototype checks.
# They describe the current queue distribution so that future runs
# have reference values to compare against.

import pandas as pd

monitor_queue = queue.copy()

# Current action mix
action_mix = (
    monitor_queue["suggested_action"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .rename("share_pct")
    .to_frame()
)

print("Current action mix:")
display(action_mix)

# Current confidence mix
confidence_mix = (
    monitor_queue["confidence"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .rename("share_pct")
    .to_frame()
)

print("\nCurrent confidence mix:")
display(confidence_mix)

# Basic data-quality checks for important numeric fields
numeric_checks = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
]

quality_rows = []

for col in numeric_checks:
    quality_rows.append({
        "feature": col,
        "missing": int(monitor_queue[col].isna().sum()),
        "min": monitor_queue[col].min(),
        "max": monitor_queue[col].max(),
    })

quality_report = pd.DataFrame(quality_rows)

print("\nBasic data-quality checks:")
display(quality_report)

assert quality_report["missing"].sum() == 0, (
    "Missing values found in monitored numeric fields."
)

# Current reference ranking metric from the validated model.
# This is a reference point, not a production threshold.
validated_precision_at_50 = 0.740

print(
    f"\nValidated Precision@50 reference: "
    f"{validated_precision_at_50:.3f}"
)

print(
    "\nMonitoring status: PASS — "
    "current queue distributions and data-quality checks were recorded "
    "for future comparison."
)

print(
    "\nImportant: no automatic retraining is triggered by this notebook. "
    "Any trigger requires investigation and fresh validation."
)

Current action mix:


,share_pct
suggested_action,
refresh_and_review_ctr,65.0
refresh,17.5
refresh_and_review_engagement,17.5



Current confidence mix:


,share_pct
confidence,
high,80.5
medium,19.5



Basic data-quality checks:


,feature,missing,min,max
0,impressions_90d,0,625.0,45127.00
1,clicks_90d,0,0.0,161.00
2,sessions_90d,0,2.0,161.00
3,avg_position,0,1.1,39.40
4,ctr,0,0.0,0.94
5,content_age_days,0,106.0,333.00
6,days_since_last_update,0,20.0,194.00



Validated Precision@50 reference: 0.740

Monitoring status: PASS — current queue distributions and data-quality checks were recorded for future comparison.

Important: no automatic retraining is triggered by this notebook. Any trigger requires investigation and fresh validation.


In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [23]:
# 5. Exports for the paper

from pathlib import Path

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Export the final recommendation queue
queue_path = OUTPUT_DIR / "validation_queue.csv"
queue.to_csv(queue_path, index=False)

print(f"Queue exported to: {queue_path}")
print(f"Rows: {len(queue):,}")
print(f"Columns: {len(queue.columns)}")

Queue exported to: work/outputs/validation_queue.csv
Rows: 200
Columns: 28


In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.